# arXiv Tools

![arXiv Tools](https://proto-bio.github.io/proto-assets/images/tool/arxiv/hero.png)

This notebook demonstrates `run_arxiv_search`, which searches arXiv by title, author, abstract, or category. Each hit carries its versioned arXiv ID, title, authors, abstract, submission dates, categories, DOI and journal reference when present, and abstract and PDF links.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("arxiv")
display_overview("arxiv")
display_docs_section("arxiv", "Background")

## Available tools

In [ ]:
display_available_tools("arxiv")

### `run_arxiv_search`

`run_arxiv_search` takes arXiv `search_query` strings built from field prefixes such as `ti:`, `au:`, `abs:`, and `cat:`. Here we search titles for protein binder design.

In [ ]:
from proto_tools import (
    ArxivSearchConfig,
    ArxivSearchInput,
    run_arxiv_search,
)

In [ ]:
# Display docs
display_api_reference("arxiv", "input", "run_arxiv_search")

# Search titles for a phrase
inputs = ArxivSearchInput(queries='ti:"protein binder"')

In [ ]:
# Display config docs
display_api_reference("arxiv", "config", "run_arxiv_search")

# Return the 5 most relevant hits | see docs above for all fields
config = ArxivSearchConfig(max_results=5)

In [ ]:
# Run the search
output = run_arxiv_search(inputs, config)

In [ ]:
# Display output docs
display_api_reference("arxiv", "output", "run_arxiv_search")

# Inspect the ranked hits
result = output.results[0]
print(f"{result.total_count} matching preprints")
for hit in result.hits:
    print(f"{hit.arxiv_id}  {hit.published[:10]}  [{hit.primary_category}]  {hit.title[:70]}")

#### Combining Fields and Sorting by Date

Field prefixes combine with `AND`, `OR`, and `ANDNOT`. Here we run two queries in one call, keeping biomolecule preprints that mention diffusion in the abstract and recent protein language model preprints, newest first. arXiv asks for at most one request every three seconds, so the tool waits between the two queries.

In [ ]:
recent_output = run_arxiv_search(
    ArxivSearchInput(queries=["cat:q-bio.BM AND abs:diffusion AND abs:protein", 'abs:"protein language model"']),
    ArxivSearchConfig(max_results=3, sort_by="submittedDate"),
)

for result in recent_output.results:
    print(f"\n{result.query}")
    for hit in result.hits:
        print(f"  {hit.arxiv_id}  {hit.published[:10]}  {hit.title[:70]}")
        print(f"    {hit.abs_url}  doi={hit.doi}")

## Export Results

Search results can be saved to JSON or CSV. arXiv metadata is CC0, so it can be stored and shared freely; this example still writes to a temporary directory to keep the repository clean.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as output_dir:
    output.export("arxiv_search", export_path=output_dir, file_format="csv")
    print(f"Exported {sorted(path.name for path in Path(output_dir).iterdir())}")